## Week 3: Forward and Backward Selection and Partial Least Squares Regression

For this week, I will be performing forward and backward selection on the datasets and then will fit the best ridge regression model for each dataset with the forward selected features and then the backward selected features. I will also perform partial least squares to see how it compares.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.cross_decomposition import PLSRegression

## Dataset 1: Instacart Dataset

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

X = data1_transformed.drop(columns = ['ordered'])
y = data1_transformed['ordered']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# Alpha value came from hyperparameter tuning in Week 2
model = Ridge(alpha = 0.1873817422860385)

forward = SequentialFeatureSelector(model, direction = 'forward')

forward.fit(X_train, y_train)

print(f'Selected Features: {forward.get_support()}')

Selected Features: [False False False False  True False  True  True  True]


In [4]:
X_train_forward = X_train[['log_frequency', 'log_Last Product Order', 'log_Last Order', 'log_Orders Since Last Purchase']]
X_test_forward = X_test[['log_frequency', 'log_Last Product Order', 'log_Last Order', 'log_Orders Since Last Purchase']]

model.fit(X_train_forward, y_train)

y_preds = model.predict(X_test_forward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.012,0.007,0.105
2,-0.002,0.008,0.130
3,0.008,0.019,0.296
4,0.018,0.020,0.318
5,0.030,0.031,0.483
6,0.047,0.045,0.703
7,0.068,0.055,0.868
8,0.091,0.075,1.187
9,0.128,0.120,1.887


In [5]:
backward = SequentialFeatureSelector(model, direction = 'backward')

backward.fit(X_train, y_train)

print(f'Selected Features: {backward.get_support()}')

Selected Features: [False False False  True  True  True  True False  True]


In [6]:
X_train_backward = X_train[['log_num_orders', 'log_frequency', 'log_Product Popularity', 'log_Last Product Order',
                            'log_Orders Since Last Purchase']]
X_test_backward = X_test[['log_num_orders', 'log_frequency', 'log_Product Popularity', 'log_Last Product Order',
                            'log_Orders Since Last Purchase']]

model.fit(X_train_backward, y_train)

y_preds = model.predict(X_test_backward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.013,0.006,0.098
2,-0.002,0.009,0.146
3,0.008,0.016,0.256
4,0.018,0.020,0.308
5,0.031,0.030,0.476
6,0.047,0.044,0.691
7,0.068,0.057,0.892
8,0.091,0.075,1.178
9,0.129,0.121,1.910


In [7]:
PLS_model = PLSRegression()

PLS_model.fit(X_train, y_train)

y_preds = PLS_model.predict(X_test)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.031,0.006,0.099
2,-0.005,0.009,0.144
3,0.011,0.012,0.189
4,0.026,0.024,0.378
5,0.041,0.031,0.494
6,0.056,0.044,0.696
7,0.073,0.060,0.950
8,0.095,0.077,1.208
9,0.130,0.118,1.849


## Dataset 2: Amazon Reviews Dataset

In [2]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked', 'rating'])
y = data2_transformed['liked']

X.head()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,False,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,True,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,True,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,True,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,True,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# Alpha value came from hyperparameter tuning in Week 2
model = Ridge(alpha = 100)

forward = SequentialFeatureSelector(model, direction = 'forward')

forward.fit(X_train, y_train)

print(f'Selected Features: {forward.get_support()}')

Selected Features: [False False False False False False False False  True False False  True
 False  True False  True  True False  True False False  True False False
  True  True  True  True  True  True  True  True  True  True  True False]


In [10]:
X_train.columns

Index(['verified_purchase', 'Beverages', 'Snacks & Sweets', 'Pantry Staples',
       'Coffee', 'Snack Foods', 'Candy & Chocolate',
       'Bottled Beverages, Water & Drink Mixes', 'Cooking & Baking',
       'Breads & Bakery', 'Single-Serve Capsules & Pods', 'Tea', 'Cookies',
       'Meat Snacks', 'Soups, Stocks & Broths', 'Nuts & Seeds', 'Jerky',
       'Juices', 'Ground Coffee', 'Breakfast Foods', 'Cereals', 'Granola',
       'Fruit & Herbal Tea', 'Herbal', 'Bars', 'Roasted Coffee Beans',
       'Whole Coffee Beans', 'Canned, Jarred & Packaged Foods', 'Cold Cereals',
       'Baking Syrups, Sugars & Sweeteners', 'Black', 'log_helpful_vote',
       'log_average_rating', 'log_price', 'log_product_review_count',
       'log_user_review_count'],
      dtype='str')

In [9]:
X_train_forward = X_train[['Breads & Bakery', 'Cookies', 'Soups, Stocks & Broths', 'Jerky', 'Juices',
                           'Breakfast Foods', 'Fruit & Herbal Tea', 'Roasted Coffee Beans', 'Whole Coffee Beans',
                           'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners',
                           'Black', 'log_helpful_vote', 'log_average_rating', 'log_price',
                           'log_product_review_count', 'log_user_review_count']]
X_test_forward = X_test[['Breads & Bakery', 'Cookies', 'Soups, Stocks & Broths', 'Jerky', 'Juices',
                           'Breakfast Foods', 'Fruit & Herbal Tea', 'Roasted Coffee Beans', 'Whole Coffee Beans',
                           'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners',
                           'Black', 'log_helpful_vote', 'log_average_rating', 'log_price',
                           'log_product_review_count', 'log_user_review_count']]

model.fit(X_train_forward, y_train)

y_preds = model.predict(X_test_forward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.811,0.733,0.878
2,0.840,0.667,0.798
3,0.850,0.786,0.940
4,0.861,0.867,1.037
5,0.870,0.929,1.111
6,0.877,0.800,0.957
7,0.884,0.857,1.026
8,0.892,0.800,0.957
9,0.905,1.000,1.197


In [8]:
backward = SequentialFeatureSelector(model, direction = 'backward')

backward.fit(X_train, y_train)

print(f'Selected Features: {backward.get_support()}')

Selected Features: [False False False False False False False False  True False False  True
 False  True False  True  True False  True False False  True False False
  True  True  True  True  True  True  True  True  True  True  True False]


In [10]:
X_train_backward = X_train[['Breads & Bakery', 'Cookies', 'Soups, Stocks & Broths', 'Jerky', 'Juices',
                           'Breakfast Foods', 'Fruit & Herbal Tea', 'Roasted Coffee Beans', 'Whole Coffee Beans',
                           'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners',
                           'Black', 'log_helpful_vote', 'log_average_rating', 'log_price',
                           'log_product_review_count', 'log_user_review_count']]

X_test_backward = X_test[['Breads & Bakery', 'Cookies', 'Soups, Stocks & Broths', 'Jerky', 'Juices',
                           'Breakfast Foods', 'Fruit & Herbal Tea', 'Roasted Coffee Beans', 'Whole Coffee Beans',
                           'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners',
                           'Black', 'log_helpful_vote', 'log_average_rating', 'log_price',
                           'log_product_review_count', 'log_user_review_count']]

model.fit(X_train_backward, y_train)

y_preds = model.predict(X_test_backward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.811,0.733,0.878
2,0.840,0.667,0.798
3,0.850,0.786,0.940
4,0.861,0.867,1.037
5,0.870,0.929,1.111
6,0.877,0.800,0.957
7,0.884,0.857,1.026
8,0.892,0.800,0.957
9,0.905,1.000,1.197


In [6]:
PLS_model = PLSRegression()

PLS_model.fit(X_train, y_train)

y_preds = PLS_model.predict(X_test)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.674,0.600,0.718
2,0.775,0.733,0.878
3,0.814,0.857,1.026
4,0.839,0.867,1.037
5,0.862,0.857,1.026
6,0.884,0.867,1.037
7,0.906,0.929,1.111
8,0.937,0.800,0.957
9,0.967,0.929,1.111


## Conclusion

Looking at the results, the biggest surprise was the selection processes selecting the same features for dataset 2. I am very curious why the models selected those 18 features, especially since most of them are a binary flag so there shouldn't be that much variation between them. It could be that there is a trend of certain categories are more liked compared to others.